In [1]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import numpy as np
import math
from scipy.stats import norm
import matplotlib.pyplot as plt
import sys

In [2]:
class LogisticsNetworkData:

    def __init__(self):
        self.df_V = pd.read_excel('fullDist Set V Data Location.xlsx')
        self.df_C = pd.read_excel('fullDist Set C Data.xlsx')
        self.df_R = pd.read_excel('fullDist R.xlsx')
        self.time_matrix = np.loadtxt("durationFULL.txt")
        self.distance_matrix = np.loadtxt("distanceFULL.txt")
        self.df_inputs = pd.read_excel('fullDistInputs.xlsx')
        self.df_K = pd.read_excel('K_ijt_data_fixed.xlsx')

        self._define_sets()
        self._define_parameters()
        self._create_arc_set()

    def _define_sets(self):
        self.C = len(self.df_C)
        self.V = len(self.df_V)
        self.V0, self.V1, self.V2, self.V3 = [], [], [], []
        self.R = len(self.df_R)

        for i in range(self.V):
            location_type = self.df_V['Type'][i]
            if location_type in ['Warehouse', 'Supplier']:
                self.V0.append(self.df_V['Index'][i])
            elif location_type == 'Shelter':
                self.V1.append(self.df_V['Index'][i])
            elif location_type == 'RSA':
                self.V2.append(self.df_V['Index'][i])
            elif location_type == 'PoD':
                self.V3.append(self.df_V['Index'][i])

        # Time Period Set
        self.tperiod = 8
        self.T = int(self.df_inputs['Hours in 6 days'][0] / self.tperiod)
        self.T_Bef = list(range(int(self.T / 2)))
        self.T_Aft = list(range(int(self.T / 2), self.T))

        # Scenario Indices
        self.sce = list(range(5))
        self.P = [1 / 5] * 5

    def _define_parameters(self):
        # Parameters
        self.W_k = self.df_inputs['People on Truck'][0]
        self.tp_salary = self.W_k * self.tperiod
        self.fuel = self.df_inputs['Fuel Cost Per Mile'][0] * 0.000621371
        self.open_array = [self.df_inputs['Open Cost V0'][0], self.df_inputs['Open Cost V1'][0], 
                           self.df_inputs['Open Cost V2'][0], self.df_inputs['Open Cost V3'][0]]

        # Shelter Capacity
        self.q_i = [self.df_V['Pallet Capacity'][i] for i in self.V1]

        # Starting Inventory
        self.Inv = [[self.df_V['S-Kit StartInv'][i], self.df_V['P-Kit StartInv'][i], 
                     self.df_V['Water StartInv'][i], self.df_V['Food StartInv'][i], 
                     self.df_V['Cot StartInv'][i], self.df_V['Blanket StartInv'][i]] 
                    for i in range(self.V)]
        self.I_ir0 = [self.Inv[i][:len(self.df_R)] for i in range(self.V)]

        # Penalty Costs
        self.p_i = [self.df_V['P_i'][i] for i in range(self.V)]

        # Travel Time and Distance
        self.t_ijt = [[math.ceil(self.time_matrix[i][j] / 3600 / self.tperiod) for j in range(self.V)] for i in range(self.V)]
        self.e_ij = [[self.distance_matrix[i][j] * 0.000621371 for j in range(self.V)] for i in range(self.V)]

        # Truck Delay
        self.h_i = [self.df_V['TruckDelay'][i] for i in range(self.V)]

        # Resource Fraction
        self.w_r = [1 / self.df_R['Pallet Units on Truck'][r] for r in range(len(self.df_R))]

        # Procurement Costs
        self.cost_ir = [[self.df_V['Water PC'][i] if r == 2 else 
                         self.df_V['Food PC'][i] if r == 3 else 0 for r in range(len(self.df_R))] for i in self.V0]

        # Shelter and PoD Pallet Factors
        self.g_shelter = [self.df_R['pallet per S-Kit'][r] for r in range(len(self.df_R))]
        self.g_pod = [self.df_R['pallet per P-Kit'][r] for r in range(len(self.df_R))]

        # Arc Capacity
        self.K_ijt = {}
        for _, row in self.df_K.iterrows():
            i, j, t, capacity = int(row['From_Node']), int(row['To_Node']), int(row['Time_Period']), row['Capacity']
            if i not in self.K_ijt:
                self.K_ijt[i] = {}
            if j not in self.K_ijt[i]:
                self.K_ijt[i][j] = {}
            self.K_ijt[i][j][t] = capacity

    def _create_arc_set(self):
        self.A = set()
        self.no_demand = []

        # Identify locations without demand
        for i in set(self.V1).union(set(self.V3)):
            i_has_demand = any(self.df_V['County'][i] == self.df_C['County'][c] and 
                               self.df_C['Shelter Total Demand'][c] > 0 for c in range(self.C))
            if not i_has_demand:
                self.no_demand.append(i)

        # Generate arcs while excluding invalid pairs
        for i in range(self.V):
            for j in range(self.V):
                if i != j and i not in self.no_demand and j not in self.no_demand:
                    if (i not in self.V1 or j not in self.V3) and (i not in self.V3 or j not in self.V1):
                        self.A.add((i, j))

In [3]:
class ScenarioData:

    def __init__(self, 
                 demand_file='demand_data_.xlsx', 
                 demand_sheets=['Demand_1', 'Demand_2', 'Demand_3', 'Demand_4', 'Demand_5'], 
                 num_scenarios=5):
        self.demand_file = demand_file
        self.demand_sheets = demand_sheets
        self.num_scenarios = num_scenarios
        self.demand_data = {}
        self.probabilities = [1 / num_scenarios] * num_scenarios

        self._load_demand_data()

    def _load_demand_data(self):

        for s, sheet in enumerate(self.demand_sheets):
            df = pd.read_excel(self.demand_file, sheet_name=sheet, header=0, index_col=0)
            df = df.fillna(0)  # Fill NaN with 0 if needed
            demand_array = df.values

            for i in range(demand_array.shape[0]):
                for t in range(demand_array.shape[1]):
                    self.demand_data[(i, t, s)] = demand_array[i, t]

    def get_demand(self, location, time, scenario):
        return self.demand_data.get((location, time, scenario), 0)

    def get_probabilities(self):
        return self.probabilities

    def get_scenarios(self):
        return list(range(self.num_scenarios))

In [4]:
class TwoStageStochasticModel:
    def __init__(self, logistics_data, scenario_data):
        self.log_data = logistics_data
        self.scenario_data = scenario_data
        self.model = gp.Model("Two-Stage Stochastic Programming Model")

        # Initialize variables, constraints, and objective
        self._define_variables()
        self._define_constraints()
        self._set_objective()

    def _define_variables(self):
        """
        Define first-stage and second-stage decision variables.
        """
        # First-stage decision variables
        self.I = self.model.addVars(
            self.log_data.V, self.log_data.R, self.log_data.T, vtype=GRB.CONTINUOUS, name="I"
        )

        # Second-stage decision variables
        self.U = self.model.addVars(
            self.log_data.V, self.log_data.T, self.scenario_data.get_scenarios(), vtype=GRB.CONTINUOUS, name="U"
        )

        self.S = self.model.addVars(
            self.log_data.V, self.log_data.T, self.scenario_data.get_scenarios(), vtype=GRB.CONTINUOUS, name="S"
        )

        self.f = self.model.addVars(
            [(i, j, r, t, s) for i, j in self.log_data.A for r in range(self.log_data.R) 
             for t in range(self.log_data.T) for s in self.scenario_data.get_scenarios()],
            vtype=GRB.CONTINUOUS,
            name="f"
        )

    def _define_constraints(self):

        # Starting inventory
        for i in range(self.log_data.V):
            for r in range(self.log_data.R):
                self.model.addConstr(
                    self.I[i, r, 0] == self.log_data.I_ir0[i][r], name=f"InitialInventory_{i}_{r}"
                )

        # Initial demand shortage
        for i in range(self.log_data.V):
            for s in self.scenario_data.get_scenarios():
                self.model.addConstr(
                    self.U[i, 0, s] == self.scenario_data.get_demand(i, 0, s),
                    name=f"InitialDemandShortage_{i}_{s}"
                )

        # Update demand shortage
        for i in range(self.log_data.V):
            for t in range(1, self.log_data.T):
                for s in self.scenario_data.get_scenarios():
                    self.model.addConstr(
                        self.U[i, t, s] == self.U[i, t - 1, s] - self.S[i, t, s] + self.scenario_data.get_demand(i, t, s),
                        name=f"DemandShortageUpdate_{i}_{t}_{s}"
                    )

        # Demand satisfaction constraint
        for i in range(self.log_data.V):
            for t in range(self.log_data.T):
                for s in self.scenario_data.get_scenarios():
                    self.model.addConstr(
                        self.S[i, t, s] <= self.U[i, t, s],
                        name=f"DemandSatisfaction_{i}_{t}_{s}"
                    )

        # Inventory update
        for i in range(self.log_data.V):
            for r in range(self.log_data.R):
                for t in range(self.log_data.T - 1):
                    for s in self.scenario_data.get_scenarios():
                        outgoing_flow = gp.quicksum(
                            self.f[i, j, r, t, s] for j in range(self.log_data.V) if (i, j) in self.log_data.A
                        )
                        incoming_flow = gp.quicksum(
                            self.f[j, i, r, t - self.log_data.t_ijt[j][i] + 1, s]
                            for j in range(self.log_data.V)
                            if (j, i) in self.log_data.A and t - self.log_data.t_ijt[j][i] + 1 >= 0
                        )

                        
                        if i in self.log_data.V1:
                            self.model.addConstr(
                                self.I[i, r, t + 1] == self.I[i, r, t] - outgoing_flow + incoming_flow -
                                self.S[i, t, s] * self.log_data.g_shelter[r],
                                name=f"InventoryUpdate_Shelter_{i}_{r}_{t}_{s}"
                            )
                        elif i in self.log_data.V3:
                            self.model.addConstr(
                                self.I[i, r, t + 1] == self.I[i, r, t] - outgoing_flow + incoming_flow -
                                self.S[i, t, s] * self.log_data.g_pod[r],
                                name=f"InventoryUpdate_Pod_{i}_{r}_{t}_{s}"
                            )
                        else:
                            self.model.addConstr(
                                self.I[i, r, t + 1] == self.I[i, r, t] - outgoing_flow + incoming_flow,
                                name=f"InventoryUpdate_{i}_{r}_{t}_{s}"
                            )

        # Inventory sufficiency constraint
        for i in range(self.log_data.V):
            for r in range(self.log_data.R):
                for t in range(self.log_data.T):
                    for s in self.scenario_data.get_scenarios():
                        outgoing_flow = gp.quicksum(
                            self.f[i, j, r, t, s] for j in range(self.log_data.V) if (i, j) in self.log_data.A
                        )
                        
                        if i in self.log_data.V1:
                            self.model.addConstr(
                                self.I[i, r, t] >= outgoing_flow + self.S[i, t, s] * self.log_data.g_shelter[r],
                                name=f"InventorySufficiency_Shelter_{i}_{r}_{t}_{s}"
                            )
                        elif i in self.log_data.V3:
                            self.model.addConstr(
                                self.I[i, r, t] >= outgoing_flow + self.S[i, t, s] * self.log_data.g_pod[r],
                                name=f"InventorySufficiency_Pod_{i}_{r}_{t}_{s}"
                            )
                        else:
                            self.model.addConstr(
                                self.I[i, r, t] >= outgoing_flow,
                                name=f"InventorySufficiency_{i}_{r}_{t}_{s}"
                            )

        # Flow capacity constraints
        for i, j in self.log_data.A:
            for t in range(self.log_data.T):
                for s in self.scenario_data.get_scenarios():
                    if i in self.log_data.K_ijt and j in self.log_data.K_ijt[i] and t in self.log_data.K_ijt[i][j]:
                        self.model.addConstr(
                            gp.quicksum(
                                self.log_data.w_r[r] * self.f[i, j, r, t, s] for r in range(self.log_data.R)
                            ) <= self.log_data.K_ijt[i][j][t],
                            name=f"FlowCapacity_{i}_{j}_{t}_{s}"
                        )

    def _set_objective(self):

        # Minimize total expected penalty costs
        objective = gp.quicksum(
            self.scenario_data.get_probabilities()[s] * gp.quicksum(
                self.log_data.p_i[i] * self.U[i, t, s] for i in range(self.log_data.V) for t in range(self.log_data.T)
            )
            for s in self.scenario_data.get_scenarios()
        )
        self.model.setObjective(objective, GRB.MINIMIZE)

    def solve(self):

        self.model.optimize()
        if self.model.status == GRB.OPTIMAL:
            print(f"Optimal Objective Value: {self.model.ObjVal}")
        else:
            print("The model did not solve to optimality. Status code:", self.model.status)

    def save_results(self, output_file="decision_variables_output.xlsx"):

        if self.model.status != GRB.OPTIMAL:
            print("No optimal solution available to save.")
            return

        # Lists to store results for each variable
        inventory_data = []
        demand_shortage_data = []
        demand_satisfaction_data = []
        flow_data = []

        # Extract Inventory (I) values
        for i in range(self.log_data.V):
            for r in range(self.log_data.R):
                for t in range(self.log_data.T):
                    inventory_data.append([i, r, t, self.I[i, r, t].X])

        # Extract Demand Shortage (U) values
        for i in range(self.log_data.V):
            for t in range(self.log_data.T):
                for s in self.scenario_data.get_scenarios():
                    demand_shortage_data.append([i, t, s, self.U[i, t, s].X])

        # Extract Demand Satisfaction (S) values
        for i in range(self.log_data.V):
            for t in range(self.log_data.T):
                for s in self.scenario_data.get_scenarios():
                    demand_satisfaction_data.append([i, t, s, self.S[i, t, s].X])

        # Extract Flow (f) values
        for i, j in self.log_data.A:
            for r in range(self.log_data.R):
                for t in range(self.log_data.T):
                    for s in self.scenario_data.get_scenarios():
                        flow_data.append([i, j, r, t, s, self.f[i, j, r, t, s].X])

        # Create a Pandas Excel writer
        with pd.ExcelWriter(output_file) as writer:
            # Convert data to DataFrame and save to Excel sheets
            pd.DataFrame(inventory_data, columns=["Node", "Commodity", "Time", "Value"]).to_excel(
                writer, sheet_name="Inventory", index=False
            )
            pd.DataFrame(demand_shortage_data, columns=["Node", "Time", "Scenario", "Value"]).to_excel(
                writer, sheet_name="DemandShortage", index=False
            )
            pd.DataFrame(demand_satisfaction_data, columns=["Node", "Time", "Scenario", "Value"]).to_excel(
                writer, sheet_name="DemandSatisfaction", index=False
            )
            pd.DataFrame(flow_data, columns=["From_Node", "To_Node", "Commodity", "Time", "Scenario", "Value"]).to_excel(
                writer, sheet_name="Flow", index=False
            )

        print(f"Decision variable results saved to '{output_file}'.")

In [5]:
if __name__ == "__main__":

    # Initialize the logistics network data
    logistics_data = LogisticsNetworkData()

    # Initialize the ScenarioData class
    scenario_data = ScenarioData()

    # Initialize the model
    model = TwoStageStochasticModel(logistics_data, scenario_data)

    # Solve the model
    model.solve()
    
    # Save the results to an Excel file
    model.save_results("stochastic_model_output.xlsx")

Set parameter Username
Academic license - for non-commercial use only - expires 2025-07-29
Gurobi Optimizer version 11.0.2 build v11.0.2rc0 (win64 - Windows 11+.0 (26100.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-13800H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 57918 rows, 768384 columns and 2316028 nonzeros
Model fingerprint: 0x2de0f594
Coefficient statistics:
  Matrix range     [9e-03, 1e+00]
  Objective range  [2e-02, 2e-01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [7e+00, 1e+04]
Presolve removed 12543 rows and 116382 columns
Presolve time: 1.62s
Presolved: 45375 rows, 652002 columns, 2048860 nonzeros

Concurrent LP optimizer: primal simplex, dual simplex, and barrier
Showing barrier log only...

Elapsed ordering time = 10s
Elapsed ordering time = 10s
Elapsed ordering time = 15s
Ordering time: 19.97s

Barrier statistics:
 AA' NZ     : 1.556e+06
 Factor NZ  : 6.533e+07 (roughly